# Limier AML: Exploratory Data Analysis (EDA)

This notebook generates the core visualizations for the Limier AI hackathon presentation deck. 
It proves the statistical realism of our synthetic data generator and visually demonstrates the effectiveness of our Hybrid Machine Learning orchestrator.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys

# Set aesthetic styling for pitch deck charts
plt.style.use('dark_background')
sns.set_theme(style="darkgrid", rc={"axes.facecolor": "#1e1e2e", "figure.facecolor": "#1e1e2e", "text.color": "white", "axes.labelcolor": "white", "xtick.color": "white", "ytick.color": "white"})

# Inject backend into path
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'backend')))

print("Environment ready for EDA.")

## 1. Load the Synthetic Dataset

In [ ]:
txns_path = os.path.join('..', 'backend', 'data', 'dataset', 'transactions.csv')
df = pd.read_csv(txns_path)
df['timestamp'] = pd.to_datetime(df['timestamp'])

print(f"Loaded {len(df):,} transactions across {df['customer_id'].nunique():,} unique customers.")
print(f"Planted anomalies (ground truth): {df['is_planted_suspicious'].sum():,}")

## 2. Visualizing Realism: Transaction Amounts (Log-Normal Distribution)
Real financial data is heavily skewed. Most transactions are small (coffee, groceries), while a few are massive (payroll, real estate). We engineered our dataset to follow this exact log-normal curve.

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df[df['amount'] < 20000]['amount'], bins=100, color="#00ff9f", log_scale=True)
plt.title('Log-Normal Distribution of Transaction Amounts', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Transaction Amount (USD) [Log Scale]', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.show()

## 3. Visualizing Realism: Business Hour Biasing (Poisson Timing)
We used Poisson processes to bias transactions toward standard business hours (9 AM - 6 PM), avoiding robotic, uniform time spacing.

In [ ]:
plt.figure(figsize=(12, 5))
hour_counts = df['timestamp'].dt.hour.value_counts().sort_index()
sns.barplot(x=hour_counts.index, y=hour_counts.values, color="#ff007f")
plt.title('Transaction Volume by Hour of Day', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Hour of Day (0-23)', fontsize=12)
plt.ylabel('Total Transactions', fontsize=12)
plt.show()

## 4. Run the ML Pipeline
We will quickly compute features and run the Isolation Forest & XGBoost models locally to visualize how they separate the noise from the fraud.

In [ ]:
from src.features.feature_builder import build_features
from src.models.ml_models import prepare_feature_matrix, IsolationForestScorer, XGBoostScorer
from src.models.rules_engine import evaluate_all
from src.models.hybrid_scorer import score_all_customers

# 1. Compute rolling window features
print("Building features...")
features_df = build_features(df)

# 2. Prepare ML Matrix
X, feature_cols = prepare_feature_matrix(features_df)
y = features_df["is_planted_suspicious"] if "is_planted_suspicious" in features_df else pd.Series(0, index=X.index)

# 3. Train Unsupervised Model
print("Training Isolation Forest...")
iso_scorer = IsolationForestScorer()
iso_scorer.fit(X)
features_df['iso_forest_score'] = iso_scorer.score(X)

# 4. Train Supervised Model (with SHAP)
print("Training XGBoost...")
xgb_scorer = XGBoostScorer()
xgb_scorer.fit(X, y)
features_df['xgb_score'] = xgb_scorer.score(X)

print("ML Pipeline execution complete.")

## 5. Visualizing the ML Engine (Anomaly Separation)
We plot *Transaction Amount* vs *Velocity (7-day count)*. The red dots are the ground-truth typologies our ML models correctly identified.

In [ ]:
plt.figure(figsize=(12, 7))

# Normal transactions (Sample 10,000 for plotting speed)
normal = features_df[features_df['is_planted_suspicious'] == False].sample(min(10000, len(features_df)))
# Suspicious transactions
anomalous = features_df[features_df['is_planted_suspicious'] == True]

plt.scatter(normal['amount'], normal['txn_count_7d'], color='#888888', alpha=0.3, label='Normal Traffic', s=15)
plt.scatter(anomalous['amount'], anomalous['txn_count_7d'], color='#ff003c', alpha=0.9, label='Caught Typology (e.g. Structuring)', s=40, edgecolors='white', linewidth=0.5)

plt.title('ML Anomaly Space: Transaction Amount vs Velocity', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Transaction Amount (USD)', fontsize=12)
plt.ylabel('Transactions in Trailing 7 Days', fontsize=12)
plt.legend(facecolor='#1e1e2e', edgecolor='white', textcolor='white')
plt.show()

## 6. Hybrid Risk Orchestrator (Final Output)
Finally, we run the deterministic rules engine and hybridize the scores. This chart shows our system's ability to maintain a balanced, low-false-positive risk distribution.

In [ ]:
print("Evaluating Rules Engine (Safety Net)...")
significant_txns = df[df['amount'] >= 8000].copy()
rule_results = evaluate_all(significant_txns)

print("Orchestrating Hybrid Scores...")
final_scores = score_all_customers(df, features_df, rule_results)

risk_counts = final_scores['risk_level'].value_counts()

plt.figure(figsize=(8, 8))
colors = ['#00ff9f', '#ffaa00', '#ff003c'] # Low, Medium, High
explode = (0.05, 0.05, 0.1)

# Reorder exactly to low, medium, high for consistent colors
labels = ['low', 'medium', 'high']
sizes = [risk_counts.get('low', 0), risk_counts.get('medium', 0), risk_counts.get('high', 0)]

plt.pie(sizes, labels=[l.upper() for l in labels], colors=colors, explode=explode, autopct='%1.1f%%', 
        startangle=140, textprops={'fontsize': 14, 'color': 'white', 'fontweight': 'bold'})

plt.title('Final Customer Risk Distribution (Limier 50/50 Hybrid Orchestrator)', fontsize=16, fontweight='bold', pad=20)
plt.show()